# ⚖️ VietLabor AI - Fine-Tuning Qwen 2.5 (7B) với Unsloth (LoRA / SFT)

Notebook này hướng dẫn tinh chỉnh (Supervised Fine-Tuning - SFT) mô hình **Qwen2.5-7B-Instruct** dựa trên tập dữ liệu **1.000 câu hỏi - đáp pháp luật lao động Việt Nam** trích xuất từ ngân hàng dữ liệu chuẩn của dự án VietLabor AI.

### 🎯 Mục tiêu huấn luyện:
1. **Chuẩn hóa phong cách trả lời**: Dạy mô hình cấu trúc phản hồi 3 bước: Kết luận trực tiếp $\rightarrow$ Phân tích điều luật $\rightarrow$ Khuyến nghị hành động.
2. **Kỷ luật căn cứ pháp lý (Citation Discipline)**: Rèn luyện mô hình viện dẫn chính xác tên điều, khoản, điểm (BLLĐ 2019, Luật BHXH, Luật ATVSLĐ, Nghị định 145/2020, Nghị định 283/2026...).
3. **Tương thích hoàn toàn với Ollama**: Xuất thẳng ra định dạng GGUF 4-bit (`q4_k_m`) để nạp về máy cá nhân chạy với Ollama.

> **Yêu cầu phần cứng**: Chạy mượt mà trên **Google Colab (GPU T4 miễn phí)** trong khoảng 15 - 20 phút.

## 1. Cài đặt Unsloth và các thư viện cần thiết

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install matplotlib seaborn datasets

## 2. Tải mô hình nền Qwen 2.5 (7B Instruct 4-bit)
Sử dụng bản 4-bit đã được Unsloth tối ưu giúp tiết kiệm 70% VRAM và tăng tốc độ huấn luyện gấp 2 lần.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Độ dài ngữ cảnh tối đa
dtype = None           # Tự động nhận diện (Float16 cho T4, Bfloat16 cho Ampere+)
load_in_4bit = True    # 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("✅ Đã tải mô hình Qwen 2.5 7B thành công!")

## 3. Cấu hình LoRA (Low-Rank Adaptation)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,     # Tối ưu hóa 0 cho Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Cấu hình LoRA hoàn tất:")
model.print_trainable_parameters()

## 4. Chuẩn bị tập dữ liệu 1.000 câu hỏi pháp luật (ChatML format)
Tải trực tiếp hoặc upload file `vietlabor_sft_1000_chatml.jsonl`.

In [ ]:
from datasets import load_dataset
import os

# Kiểm tra nếu file có sẵn trên Colab hoặc trong thư mục dự án
dataset_path = "vietlabor_sft_1000_chatml.jsonl"
alt_path = "data/training/vietlabor_sft_1000_chatml.jsonl"

if not os.path.exists(dataset_path) and os.path.exists(alt_path):
    dataset_path = alt_path

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"⚠️ Không tìm thấy file {dataset_path}. Vui lòng upload file từ data/training/vietlabor_sft_1000_chatml.jsonl vào Colab."
    )

dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"✅ Đã nạp thành công {len(dataset)} mẫu câu hỏi - đáp pháp luật!")

# Áp dụng template ChatML chuẩn của Qwen 2.5
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Mẫu dữ liệu đầu tiên sau khi format:")
print(dataset[0]["text"][:400] + "...")


## 5. Tiến hành Huấn Luyện (Training Loop)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can be True for faster training
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,               # ~2.5 - 3 Epochs cho 1000 mẫu
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "vietlabor_outputs",
    ),
)

print("🚀 Bắt đầu huấn luyện...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện hoàn tất!")

## 6. Vẽ biểu đồ hàm mất mát (Training Loss Curve)
Dùng đồ thị này để chèn trực tiếp vào báo cáo đồ án tốt nghiệp.

In [ ]:
import matplotlib.pyplot as plt

# Trích xuất log lịch sử
history = trainer.state.log_history
steps = [h["step"] for h in history if "loss" in h]
losses = [h["loss"] for h in history if "loss" in h]

plt.figure(figsize=(9, 5))
plt.plot(steps, losses, marker='o', color='#A5121B', linewidth=2, label='Training Loss')
plt.title('VietLabor AI - Quá trình suy giảm hàm mất mát (Training Loss Curve)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Bước huấn luyện (Step)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('vietlabor_training_loss.png', dpi=300)
plt.show()
print("✅ Đã lưu biểu đồ vào file vietlabor_training_loss.png")

## 7. Thử nghiệm suy diễn mô hình (Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là VietLabor AI - Trợ lý pháp lý chuyên sâu về pháp luật lao động Việt Nam. Hãy giải đáp câu hỏi của người dùng một cách chính xác, mạch lạc và luôn viện dẫn cụ thể căn cứ pháp luật."},
    {"role": "user", "content": "Công ty giữ bằng đại học bản gốc của tôi để bảo đảm hợp đồng thì có đúng luật không?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 300, use_cache = True, temperature = 0.2)
response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

print("🤖 CÂU TRẢ LỜI CỦA MÔ HÌNH SAU KHI FINE-TUNE:")
print(response)

## 8. Xuất mô hình sang GGUF để sử dụng với Ollama trên máy cục bộ
Chạy lệnh sau để Unsloth tự động nén mô hình thành file `.gguf` chuẩn `q4_k_m`.

In [ ]:
# Xuất file GGUF quantization q4_k_m
model.save_pretrained_gguf("vietlabor_model", tokenizer, quantization_method = "q4_k_m")
print("✅ Đã xuất mô hình GGUF thành công!")
print("Tải file 'vietlabor_model-unsloth.Q4_K_M.gguf' về máy cá nhân để nạp vào Ollama.")

## 9. Hướng dẫn nạp vào Ollama trên máy tính của bạn

Sau khi tải file `.gguf` về máy tính (ví dụ đặt tại thư mục dự án `d:\chatbot-law\`):

1. Tạo một file tên là `Modelfile` với nội dung:
```dockerfile
FROM ./vietlabor_model-unsloth.Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER top_p 0.9
```

2. Mở PowerShell hoặc Terminal gõ:
```bash
ollama create vietlabor-expert:7b -f Modelfile
```

3. Chạy chatbot với mô hình vừa train:
```powershell
$env:OLLAMA_MODEL="vietlabor-expert:7b"
python -m app.main
```